In [ ]:
import logging
from pathlib import Path

import cartopy
import matplotlib.pyplot as plt
import xarray
import yaml
from metplotpy.contributed.fv3_physics_tend import (
    cross_section_vert,
    physics_tend,
    planview_fv3,
    vert_profile_fv3,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True
)  # Change the level as needed

In [ ]:
METPLOTPY_BASE = Path("/glade/derecho/scratch/ahijevyc/METplotpy")
PHYSICS_TEND_DIR = METPLOTPY_BASE / "metplotpy/contributed/fv3_physics_tend"
TEST_FILE_DIR = Path("/glade/campaign/mmm/parc/ahijevyc/METplotpy")

In [ ]:
# More recent history file (residual is zero)
config = yaml.load(
    open(METPLOTPY_BASE / "test/fv3_physics_tend/tmp_500hPa.yaml", encoding="utf8"),
    Loader=yaml.FullLoader,
)
historyfile = TEST_FILE_DIR / "model_applications/miscellaneous/fv3_physics_tendency/fv3_history.nc"
gridfile = TEST_FILE_DIR / "model_applications/miscellaneous/fv3_physics_tendency/grid_spec.nc"
ds = physics_tend.prepare_ds(config, historyfile, gridfile)
ds

In [ ]:
subplot_kws = {
    "projection": cartopy.crs.LambertConformal(central_longitude=-97.6, central_latitude=35.4)
}
plt.clf()
pcm = ds.tmp.isel(time=0, pfull=-1).plot.pcolormesh(
    x="lont",
    y="latt",
    transform=cartopy.crs.PlateCarree(),
    cbar_kwargs={"shrink": 0.8},
    subplot_kws=subplot_kws,
)
physics_tend.add_conus_features(pcm.axes)
plt.show()

In [ ]:
ds["pfull"].attrs["units"]

In [ ]:
pcm = planview_fv3.planview(
    config,
    ds,
    pfull=[500],
    robust=True,
    shp=None,
    twindow=1,
    validtime="20190615T20",
)

In [ ]:
pcm = planview_fv3.planview(
    config,
    ds,
    pfull=[1000, 925, 850, 700, 500, 300, 200, 100, 0],
    robust=True,
    shp=None,
    twindow=1,
    validtime="20190615T20",
)

In [ ]:
pcm = vert_profile_fv3.vert_profile(
    config,
    ds,
    shp="shapefiles/MID_CONUS",
    twindow=1,
    xmin=-0.0002,
    xmax=0.0002,
    ofile=METPLOTPY_BASE / "docs/Users_Guide/figure/tmp.vert_profile.MID_CONUS.png",
)

In [ ]:
pcm = cross_section_vert.cross_section_vert(
    config,
    ds,
    statevarname="ugrd",
    twindow=1,
)

In [ ]:
config = yaml.load(
    open(
        METPLOTPY_BASE / "test/fv3_physics_tend/UFS.yaml",
        encoding="utf8",
    ),
    Loader=yaml.FullLoader,
)
ds = physics_tend.prepare_ds(
    config,
    TEST_FILE_DIR / f"model_applications/miscellaneous/fv3_physics_tendency/fv3_history2d.tile5.nc",
    TEST_FILE_DIR / f"model_applications/miscellaneous/fv3_physics_tendency/grid_spec.tile5.nc",
)

In [ ]:
pcm = planview_fv3.planview(
    config,
    ds,
    pfull=[500],
    statevarname="u",
)

In [ ]:
pcm = (
    ds
    .t
    .sel(pfull=500)
    .isel(time=0)
    .plot.pcolormesh(
        x="lont",
        y="latt",
        transform=cartopy.crs.PlateCarree(),
        cbar_kwargs={"shrink": 0.8},
        subplot_kws=subplot_kws,
        figsize=(10, 10),
    )
)
physics_tend.add_conus_features(pcm.axes)
plt.show()

In [ ]:
# Cutoff lows and May's UFS


# stitch tile3 (north pole) above tile5 (Americas)
def stitch_tiles(t3, t5):
    # add size of grid_yt dimension to coordinate values of t5 grid_yt.
    t5 = t5.assign_coords(grid_yt=t5.grid_yt + t5.grid_yt.size)
    # Ensure "filename" is removed from all levels of attributes
    for ds in [t3, t5]:
        ds.attrs.pop("filename", None)  # Remove from dataset attributes
        for var in ds.variables:
            ds[var].attrs.pop("filename", None)  # Remove from variable attributes
    c = xarray.concat([t3, t5], dim="grid_yt", combine_attrs="no_conflicts")
    return c


CAMPAIGN = Path("/glade/campaign/mmm/parc/mwong/ufs-mrw")
grid_spec3 = xarray.open_dataset(CAMPAIGN / "grid/grid_spec.tile3.nc")
grid_spec5 = xarray.open_dataset(CAMPAIGN / "grid/grid_spec.tile5.nc")

gds = stitch_tiles(grid_spec3, grid_spec5)

lont = gds[config["lon_name"]]
latt = gds[config["lat_name"]]
t3 = physics_tend.get_fv3ds(
    config,
    CAMPAIGN / "2020040512.F240.C768/fv3_history2d.tile3.nc",
)
t5 = physics_tend.get_fv3ds(
    config,
    CAMPAIGN / "2020040512.F240.C768/fv3_history2d.tile5.nc",
)
print("stitch history")
ds = stitch_tiles(t3, t5)

ds = ds.assign_coords(lont=lont, latt=latt)

In [ ]:
subplot_kws = {
    "projection": cartopy.crs.Stereographic(
        central_longitude=-100,
        central_latitude=45,
    )
}
plt.clf()

da = (
    ds.z500.drop_vars(["grid_yt", "grid_xt"])
    .rename({"grid_yt": "y", "grid_xt": "x"})
    .isel(time=0)
    .metpy.assign_crs(
        grid_mapping_name="stereographic",
        central_longitude=-100,
        central_latitude=45,
    )
    # .metpy.assign_y_x(force=True, tolerance=5e7 * units.m)
)

pcm = da.plot.pcolormesh(
    x="lont",
    y="latt",
    transform=cartopy.crs.PlateCarree(),
    cbar_kwargs={"shrink": 0.8},
    subplot_kws=subplot_kws,
    figsize=(10, 10),
)
physics_tend.add_conus_features(pcm.axes)
plt.show()

In [ ]:
pcm = planview_fv3.planview(
    config,
    ds,
    pfull=[500],
    robust=True,
    shp=None,
    statevarname="u",
)

In [ ]:
pcm = cross_section_vert.cross_section_vert(
    config,
    ds,
    statevarname="u",
    startpt=[34, -122],
    endpt=[33, -76],
)

In [ ]:
# another pair of forecast tiles
# This time with only 24-hours lead time
t3 = physics_tend.get_fv3ds(
    config,
    CAMPAIGN / f"2020040812.F024.C768/fv3_history2d.tile3.nc",
)
t5 = physics_tend.get_fv3ds(
    config,
    CAMPAIGN / f"2020040812.F024.C768/fv3_history2d.tile5.nc",
)
logging.info("stitch history")
ds2 = stitch_tiles(t3, t5)
ds2 = ds2.assign_coords(lont=lont, latt=latt)

pcm = planview_fv3.planview(
    config,
    ds2,
    pfull=[500],
    robust=True,
    shp=None,
    validtime=None,
)

In [ ]:
diff = ds2 - ds
# restore long_name attribute after subtraction
for da in diff:
    diff[da].attrs = ds[da].attrs
diff = diff.assign_coords(ds2.coords)
diff["area"] = gds["area"]

pcm = planview_fv3.planview(
    config,
    diff,
    pfull=[500],
    robust=True,
    shp=None,
    validtime=None,
)
pcm.fig.suptitle(f"{ds2.attrs['title']}-{ds.attrs['title']}\n{pcm.fig.get_suptitle()}")

In [ ]:
pcm = vert_profile_fv3.vert_profile(
    config, diff, shp="shapefiles/MID_CONUS", xmin=-1e-4, xmax=1.2e-4
)